### Overview

Extracts survival data of TCGA-BRCA samples for survival analysis.

In [1]:
import pandas as pd
import numpy as np

In [2]:
# import tcga brca clinical data
tcga_metadata = pd.read_csv("C:/Users/User/Documents/master_thesis_organized/datasets/TCGA_BRCA/TCGA_BRCA_cleaned_data/cleaned_sample info.csv",
                            header=0, index_col=0)

In [7]:
# check survival status counts
print(tcga_metadata['vital_status'].value_counts())
print(tcga_metadata['paper_vital_status'].value_counts())
print(tcga_metadata['vital_status'].tolist() == tcga_metadata['paper_vital_status'].tolist())

vital_status
Alive    888
Dead     143
Name: count, dtype: int64
paper_vital_status
Alive    888
Dead     143
Name: count, dtype: int64
True


In [8]:
# check race
tcga_metadata['race'].value_counts()

race
white                               708
black or african american           169
not reported                         93
asian                                60
american indian or alaska native      1
Name: count, dtype: int64

In [12]:
tcga_metadata['paper_BRCA_Subtype_PAM50'].value_counts()

paper_BRCA_Subtype_PAM50
LumA     555
LumB     209
Basal    185
Her2      82
Name: count, dtype: int64

In [11]:
# check race and subtype
print(tcga_metadata.loc[tcga_metadata['paper_BRCA_Subtype_PAM50'] == 'LumA', 'race'].value_counts())
print(tcga_metadata.loc[tcga_metadata['paper_BRCA_Subtype_PAM50'] == 'LumB', 'race'].value_counts())
print(tcga_metadata.loc[tcga_metadata['paper_BRCA_Subtype_PAM50'] == 'Her2', 'race'].value_counts())
print(tcga_metadata.loc[tcga_metadata['paper_BRCA_Subtype_PAM50'] == 'Basal', 'race'].value_counts())

race
white                        431
black or african american     62
not reported                  41
asian                         21
Name: count, dtype: int64
race
white                        131
not reported                  34
black or african american     28
asian                         16
Name: count, dtype: int64
race
white                               39
asian                               16
black or african american           16
not reported                        10
american indian or alaska native     1
Name: count, dtype: int64
race
white                        107
black or african american     63
not reported                   8
asian                          7
Name: count, dtype: int64


In [29]:
# check missing values in days to follow up and days to death
print(tcga_metadata['days_to_last_follow_up'].isna().sum())
print(tcga_metadata['days_to_death'].isna().sum())
print(tcga_metadata['paper_days_to_last_followup'].isna().sum())
print(tcga_metadata['paper_days_to_death'].isna().sum())

99
889
99
889


In [39]:
# check
tcga_metadata.loc[tcga_metadata['days_to_last_follow_up'] != tcga_metadata['paper_days_to_last_followup'], 
['days_to_last_follow_up', 'paper_days_to_last_followup', 'days_to_death', 'paper_days_to_death', 'vital_status']]

,days_to_last_follow_up,paper_days_to_last_followup,days_to_death,paper_days_to_death,vital_status
TCGA-AQ-A0Y5-01A-11R-A14M-07,NaN,NaN,172.0,172.0,Dead
TCGA-AO-A0J5-01A-11R-A034-07,735.0,705.0,792.0,792.0,Dead
TCGA-B6-A0WV-01A-11R-A109-07,NaN,NaN,2417.0,2417.0,Dead
TCGA-E2-A1LE-01A-12R-A19W-07,NaN,NaN,879.0,879.0,Dead
TCGA-A2-A0CO-01A-13R-A22K-07,1468.0,3409.0,1468.0,3492.0,Dead
...,...,...,...,...,...
TCGA-D8-A1XJ-01A-11R-A14M-07,664.0,663.0,NaN,NaN,Alive
TCGA-B6-A0IG-01A-11R-A034-07,NaN,NaN,4456.0,4456.0,Dead
TCGA-LL-A6FP-01A-11R-A31O-07,0.0,677.0,NaN,NaN,Alive
TCGA-A8-A06U-01A-11R-A00Z-07,NaN,NaN,883.0,883.0,Dead


In [53]:
# extract days to follow up, days to death and vital status
tcga_survival = tcga_metadata.loc[:, ['vital_status', 'days_to_last_follow_up', 'days_to_death']]
tcga_survival.head(3)

,vital_status,days_to_last_follow_up,days_to_death
TCGA-D8-A146-01A-31R-A115-07,Alive,643.0,NaN
TCGA-AQ-A0Y5-01A-11R-A14M-07,Dead,NaN,172.0
TCGA-C8-A274-01A-11R-A16F-07,Alive,508.0,NaN


In [54]:
# create a column called overall_survival_days and make it same as days_to_death
tcga_survival['overall_survival_days'] = tcga_survival['days_to_death']
tcga_survival.head(20)

,vital_status,days_to_last_follow_up,days_to_death,overall_survival_days
TCGA-D8-A146-01A-31R-A115-07,Alive,643.0,NaN,NaN
TCGA-AQ-A0Y5-01A-11R-A14M-07,Dead,NaN,172.0,172.0
TCGA-C8-A274-01A-11R-A16F-07,Alive,508.0,NaN,NaN
TCGA-BH-A0BD-01A-11R-A034-07,Alive,554.0,NaN,NaN
TCGA-B6-A1KC-01B-11R-A157-07,Alive,1326.0,NaN,NaN
TCGA-AO-A0J5-01A-11R-A034-07,Dead,735.0,792.0,792.0
TCGA-BH-A0B1-01A-12R-A056-07,Alive,1148.0,NaN,NaN
TCGA-A2-A0YM-01A-11R-A109-07,Alive,965.0,NaN,NaN
TCGA-AO-A03N-01B-11R-A10J-07,Alive,2031.0,NaN,NaN
TCGA-E2-A1LI-01A-12R-A157-07,Alive,3121.0,NaN,NaN


In [55]:
# if there are missing values in overall_survival days, make it same as days_to_last_follow_up
tcga_survival.loc[tcga_survival['overall_survival_days'].isna(), 'overall_survival_days'] = tcga_survival.loc[tcga_survival['overall_survival_days'].isna(),'days_to_last_follow_up']
tcga_survival.head(20)

,vital_status,days_to_last_follow_up,days_to_death,overall_survival_days
TCGA-D8-A146-01A-31R-A115-07,Alive,643.0,NaN,643.0
TCGA-AQ-A0Y5-01A-11R-A14M-07,Dead,NaN,172.0,172.0
TCGA-C8-A274-01A-11R-A16F-07,Alive,508.0,NaN,508.0
TCGA-BH-A0BD-01A-11R-A034-07,Alive,554.0,NaN,554.0
TCGA-B6-A1KC-01B-11R-A157-07,Alive,1326.0,NaN,1326.0
TCGA-AO-A0J5-01A-11R-A034-07,Dead,735.0,792.0,792.0
TCGA-BH-A0B1-01A-12R-A056-07,Alive,1148.0,NaN,1148.0
TCGA-A2-A0YM-01A-11R-A109-07,Alive,965.0,NaN,965.0
TCGA-AO-A03N-01B-11R-A10J-07,Alive,2031.0,NaN,2031.0
TCGA-E2-A1LI-01A-12R-A157-07,Alive,3121.0,NaN,3121.0


In [63]:
# check the type of overall_survival_days and check if there are missing values
print(tcga_survival['overall_survival_days'].dtype)
print(tcga_survival['overall_survival_days'].isna().any())

float64
False


In [65]:
# create another column called overall_survival_years which is overall_survival_days divided by 365.25
tcga_survival['overall_survival_years'] = round(tcga_survival['overall_survival_days']/365.25,3)
tcga_survival.head(5)

,vital_status,days_to_last_follow_up,days_to_death,overall_survival_days,overall_survival_years
TCGA-D8-A146-01A-31R-A115-07,Alive,643.0,NaN,643.0,1.760
TCGA-AQ-A0Y5-01A-11R-A14M-07,Dead,NaN,172.0,172.0,0.471
TCGA-C8-A274-01A-11R-A16F-07,Alive,508.0,NaN,508.0,1.391
TCGA-BH-A0BD-01A-11R-A034-07,Alive,554.0,NaN,554.0,1.517
TCGA-B6-A1KC-01B-11R-A157-07,Alive,1326.0,NaN,1326.0,3.630


In [68]:
# check the max in overall_survival_years
tcga_survival.loc[tcga_survival['overall_survival_years'].idxmax(),:]

vital_status               Alive
days_to_last_follow_up    8605.0
days_to_death                NaN
overall_survival_days     8605.0
overall_survival_years    23.559
Name: TCGA-B6-A0RU-01A-11R-A084-07, dtype: object

In [69]:
# import tcga counts and subtype data
tcga_subtype_counts = pd.read_csv("C:/Users/User/Documents/master_thesis_organized/datasets/TCGA_BRCA/TCGA_BRCA_cleaned_data/tcga_raw_subtype_pam50genes_whole.csv", 
                       header=0, index_col=0)

In [70]:
# check if the indices match
tcga_subtype_counts.index.equals(tcga_survival.index)

True

In [71]:
# check if the subtypes match
tcga_subtype_counts['subtype'].tolist() == tcga_metadata['paper_BRCA_Subtype_PAM50'].tolist()

True

In [73]:
# merge survival and subtype data
tcga_survival_subtype = tcga_survival.join(tcga_subtype_counts.loc[:, 'subtype'])
tcga_survival_subtype.head(4)

,vital_status,days_to_last_follow_up,days_to_death,overall_survival_days,overall_survival_years,subtype
TCGA-D8-A146-01A-31R-A115-07,Alive,643.0,NaN,643.0,1.760,LumA
TCGA-AQ-A0Y5-01A-11R-A14M-07,Dead,NaN,172.0,172.0,0.471,LumA
TCGA-C8-A274-01A-11R-A16F-07,Alive,508.0,NaN,508.0,1.391,LumB
TCGA-BH-A0BD-01A-11R-A034-07,Alive,554.0,NaN,554.0,1.517,LumB


In [82]:
# there is a sample with a negative value for overall survival years, check how many there are
# remove this sample during survival analysis
print((tcga_survival_subtype['overall_survival_years'] < 0).sum())
print(tcga_survival_subtype.loc[tcga_survival_subtype['overall_survival_years'] < 0, :])

1
                             vital_status  days_to_last_follow_up  \
TCGA-PL-A8LV-01A-21R-A41B-07        Alive                    -7.0   

                              days_to_death  overall_survival_days  \
TCGA-PL-A8LV-01A-21R-A41B-07            NaN                   -7.0   

                              overall_survival_years subtype  
TCGA-PL-A8LV-01A-21R-A41B-07                  -0.019   Basal  


In [75]:
# # save the file
# tcga_survival_subtype.to_csv('tcga_brca_survival_subtype.csv')